In [0]:
%pip install gspread google-auth pandas
dbutils.library.restartPython()

In [0]:
import gspread
from google.oauth2.service_account import Credentials
import json

KEY_FILE_PATH = "/Workspace/Users/pakhei_tsang@next.co.uk/advance-mantis-398714-2168c9162641.json"

with open(KEY_FILE_PATH, "r") as f:
    creds_dict = json.load(f)

creds = Credentials.from_service_account_info(
    creds_dict, 
    scopes=["https://www.googleapis.com/auth/spreadsheets"]
)
gc = gspread.authorize(creds)

SHEET_ID = "1dsomIdRf4vlTAfoMQ-rIq1brARZmUUVTjUEmXU51BBk"
sh = gc.open_by_key(SHEET_ID)

print("Connected to:", sh.title)

In [0]:
# --- Work areas, one volume column each ---------------------------------
# The single source of truth for this notebook: the Data tab's header, the SQL
# and the row appended to the sheet are all built from this list, so adding an
# area here is the only edit any of them needs.
#
# Each entry is the matching report from the main "Elmsall Live Productivity"
# notebook - the same area code and the same `extra` predicate - narrowed to
# the event type(s) that notebook's PROC_AREAS names as the area's VOLUME
# event. A report can earn standard hours from EVERY event type in its area
# while only one of them counts as volume (Sorter 6 Packing, both BPP areas and
# the three Online Picking areas are all like this), so the event list here is
# often shorter than the report's own. Keep the two notebooks in step: an area
# added there stays invisible on this sheet until it is added here too.
#
# ORDER MATTERS - it is the column order on the Data tab. The first nine are
# the columns this sheet already had, in the order it already had them, so
# every row written before today still lines up under its own headings; the ten
# added below extend the sheet to the right.
def _zone_pred(*zones):
    """Online Picking's zone filter, lifted from the main notebook.

    Online Picking carries its zone inside a location string rather than in an
    attribute of its own: "location:XH117B" decodes as zone X, aisle H, bay
    117, level B. So a zone filter is a test on the FIRST CHARACTER after the
    key, not a match on a whole attribute value.

    Matched case-insensitively, and the prefix is stripped by cutting at the
    first colon rather than by a fixed length. Both matter: the real values are
    lower case with no space after the colon, so 'Location: %' matched none of
    them and these three reports returned nothing at all rather than too little.
    """
    _z = ", ".join(f"'{z}'" for z in zones)
    return ("AND EXISTS(PAYLOAD_ATTRIBUTES, x -> lower(x) LIKE 'location:%' AND "
            "upper(substring(trim(regexp_replace(x, '^[^:]*:', '')), 1, 1)) "
            f"IN ({_z}))")


VOLUME_AREAS = [
    {"column": "Vol_OSR_PiE",
     "area": "('Pick','Pie Station')", "events": "('MSKU','PSKU')", "extra": ""},

    {"column": "Vol_OSR_Topup",
     "area": "('Topup','TopUp','PiOrQi')", "events": "('TPUT')", "extra": ""},

    {"column": "Vol_E3_Packing",
     "area": "('E3 - Packing')", "events": "('PackingItemScannedEvent')", "extra": ""},

    # No area filter: this event type only ever comes from parcel sortation.
    {"column": "Vol_Parcel_Sortation",
     "area": None, "events": "('ParcelSortedToSack')", "extra": ""},

    {"column": "Vol_Parcel_Induct",
     "area": "('Parcel Induct')", "events": "('SPAR')", "extra": ""},

    {"column": "Vol_Inbound_Decanting",
     "area": "('Inbound Decanting')", "events": "('DECN')", "extra": ""},

    {"column": "Vol_OSR_Decanting",
     "area": "('OSR Decanting')", "events": "('ODEC')", "extra": ""},

    # BCR and E1/E2 Inducting are the same area code and the same event type,
    # told apart only by what the attributes say the stock is.
    {"column": "Vol_BCR_Inducting",
     "area": "('Induct from E1/E2')", "events": "('SPOS')",
     "extra": "AND EXISTS(PAYLOAD_ATTRIBUTES, x -> x LIKE '%RET%')"},

    {"column": "Vol_E1_E2_Inducting",
     "area": "('Induct from E1/E2')", "events": "('SPOS')",
     "extra": "AND EXISTS(PAYLOAD_ATTRIBUTES, x -> x LIKE '%PIE4EDW%')"},

    # --- Everything below is new to this sheet. ---
    {"column": "Vol_Sorter_6_Packing",
     "area": "('Sorter 6 - Packing')", "events": "('PackingItemScannedEvent')", "extra": ""},

    # Online Picking is ONE source split three ways by zone: same warehouse,
    # same area code, different zones decoded out of the location string.
    {"column": "Vol_Online_Picking_Drive",
     "area": "('Online Picking')", "events": "('PickItemEvent')",
     "extra": _zone_pred('D', 'F', 'Q', 'S', 'T', 'V')},

    {"column": "Vol_Online_Picking_Way",
     "area": "('Online Picking')", "events": "('PickItemEvent')",
     "extra": _zone_pred('A', 'B', 'E', 'G', 'J', 'L', 'X', 'W', 'R')},

    {"column": "Vol_Online_Picking_E3",
     "area": "('Online Picking')", "events": "('PickItemEvent')",
     "extra": _zone_pred('H', 'C')},

    # The two BPP areas scan items into cartons under different event names.
    {"column": "Vol_E3_BPP",
     "area": "('E3 BPP Packing')", "events": "('Scan Item to Carton')", "extra": ""},

    {"column": "Vol_E1_E2_BPP",
     "area": "('BPP')", "events": "('BppScanItemToCarton')", "extra": ""},

    # Returns processing. RSPS and ISPS share event types and are told apart by
    # area alone, so each pair needs BOTH filters - an event filter on its own
    # would merge them.
    {"column": "Vol_RSPS_Top_Up",
     "area": "('ReturnsBuffer')", "events": "('TPUT')", "extra": ""},

    {"column": "Vol_RSPS_Pick",
     "area": "('ReturnsBuffer')", "events": "('RPUT')", "extra": ""},

    {"column": "Vol_ISPS_Top_Up",
     "area": "('RSPS2Messaging')", "events": "('TPUT')", "extra": ""},

    {"column": "Vol_ISPS_Pick",
     "area": "('RSPS2Messaging')", "events": "('RPUT')", "extra": ""},
]

HEADER = ['Date', 'Hour'] + [a['column'] for a in VOLUME_AREAS]


def _volume_pred(a):
    """One area's config -> a standalone SQL boolean.

    `extra` is written starting with "AND " so it reads exactly as it does in
    the main notebook's `reports` list; the prefix is stripped here rather than
    the two notebooks spelling the same predicate two different ways.
    """
    parts = []
    if a["area"]:
        parts.append(f"PAYLOAD_AREACODE IN {a['area']}")
    parts.append(f"PAYLOAD_EVENTTYPE IN {a['events']}")
    extra = a["extra"].strip()
    if extra:
        if extra[:4].upper() == "AND ":
            extra = extra[4:].strip()
        parts.append(extra)
    return " AND ".join(f"({p})" for p in parts)


print(f"{len(VOLUME_AREAS)} work areas -> {len(HEADER)} columns")

In [0]:
# The header is written from HEADER rather than kept by hand on the tab, and
# rewritten whenever it disagrees. This notebook went from 9 volume columns to
# 19, and a tab still carrying the old header would have put ten columns of
# figures under no heading at all - the one mistake that mis-reads a sheet
# silently rather than failing.
#
# Rows written before today keep their values under the first nine columns and
# have nothing in the ten added today, which is the honest record: those areas
# were not being collected then.
try:
    ws = sh.worksheet("Data")
except gspread.exceptions.WorksheetNotFound:
    ws = sh.add_worksheet(title="Data", rows=2000, cols=len(HEADER))
    print("Created 'Data' tab")

# A row wider than the grid is rejected outright ("exceeds grid limits"), so
# the tab is widened before anything is written to it. A no-op once it has run.
if ws.col_count < len(HEADER):
    ws.resize(cols=len(HEADER))
    print(f"Widened 'Data' to {len(HEADER)} columns")

current_header = ws.row_values(1)
if current_header != HEADER:
    ws.update("A1", [HEADER], value_input_option='RAW')
    print(f"Header rewritten ({len(current_header)} -> {len(HEADER)} columns)")
else:
    print(f"Header already matches ({len(HEADER)} columns)")

In [0]:
df = (spark.read
      .format("delta")
      .load("abfss://landing@whsanalyticsdlsprodeuw.dfs.core.windows.net/streaming/landing_bonushub_event_parsed/delta/"))

df.createOrReplaceTempView("landing_bonus_hub_event_parsed")

In [0]:
# ============================================================
# PREVIOUS HOUR'S VOLUME -> one row, one column per work area.
#
# One SUM per area, each testing its own predicate, rather than a single CASE
# labelling every event with the one area it belongs to. With 19 areas that
# labelling no longer holds: an Online Picking event carrying two locations in
# different zones belongs to two of these areas, and the main notebook counts
# it in both (it tags each event with the SET of reports it matches, via array
# + explode). A first-match CASE would quietly hand it to whichever area
# happened to be listed first. Summing each area independently keeps the two
# notebooks agreeing, and needs no assumption that the filters are disjoint.
#
# The CTE carries the raw PAYLOAD_ columns through under their own names, so
# the generated predicates read exactly as they do in the main notebook.
# ============================================================
sum_exprs = ",\n    ".join(
    f"SUM(CASE WHEN {_volume_pred(a)} THEN qty ELSE 0 END) AS {a['column']}"
    for a in VOLUME_AREAS
)

# Events matching no area at all are dropped in the CTE, so an hour with events
# but none in any tracked area still reports "no volume data" rather than a row
# of zeroes. This is the job the old `WHERE work_area IS NOT NULL` did.
match_any = " OR ".join(f"({_volume_pred(a)})" for a in VOLUME_AREAS)

hourly_volume_query = f"""
WITH base AS (
    SELECT
        date_format(from_utc_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP),'Europe/London'),'yyyy-MM-dd') AS Date,
        hour(from_utc_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP),'Europe/London')) AS Hour,
        PAYLOAD_AREACODE   AS PAYLOAD_AREACODE,
        PAYLOAD_EVENTTYPE  AS PAYLOAD_EVENTTYPE,
        PAYLOAD_ATTRIBUTES AS PAYLOAD_ATTRIBUTES,
        PAYLOAD_QUANTITY   AS qty
    FROM landing_bonus_hub_event_parsed
    WHERE TRIM(PAYLOAD_WAREHOUSECODE) = 'X'
      -- Dynamically filter for the exact previous hour in local UK time
      AND hour(from_utc_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP), 'Europe/London')) = hour(from_utc_timestamp(current_timestamp() - INTERVAL 1 HOUR, 'Europe/London'))
      AND to_date(from_utc_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP), 'Europe/London')) = to_date(from_utc_timestamp(current_timestamp() - INTERVAL 1 HOUR, 'Europe/London'))
      AND ( {match_any} )
)
SELECT 
    Date,
    Hour,
    {sum_exprs}
FROM base
GROUP BY Date, Hour
"""

# Execute query and convert to Pandas
hourly_volumes_df = spark.sql(hourly_volume_query)
pdf = hourly_volumes_df.toPandas().fillna(0)

# Append to the Google Sheet
if pdf.empty:
    print("No volume data found for the previous hour.")
else:
    # Ordered by HEADER rather than by whatever order toPandas hands back, so
    # the figures cannot land under the wrong headings.
    pdf = pdf[HEADER]
    values_to_append = pdf.values.tolist()
    ws.append_rows(values_to_append, value_input_option='USER_ENTERED')
    print(f"Successfully appended {len(values_to_append)} row(s) for Hour {pdf['Hour'].iloc[0]} to the 'Data' tab.")

    # Every area is named, including the ones that reported nothing. An area
    # can genuinely be idle for an hour, so a zero is not an error - but an
    # area reading zero hour after hour, or zero while every other area is
    # busy, is a filter that has stopped matching its source, and a log listing
    # only the areas that reported would look exactly like a healthy run.
    row = pdf.iloc[0]
    for a in VOLUME_AREAS:
        print(f"  {a['column']:<26} {row[a['column']]}")
    idle = [a['column'] for a in VOLUME_AREAS if not float(row[a['column']])]
    if idle:
        print(f"  no volume from {len(idle)} of {len(VOLUME_AREAS)} area(s): "
              + ", ".join(idle))